In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
selected_sensors = ["s_7", "s_12", "s_14", "s_20", "s_21"]

feature_cols = (
    selected_sensors +
    [f"{s}_roll_mean" for s in selected_sensors] +
    [f"{s}_roll_std" for s in selected_sensors]
)

feature_cols

['s_7',
 's_12',
 's_14',
 's_20',
 's_21',
 's_7_roll_mean',
 's_12_roll_mean',
 's_14_roll_mean',
 's_20_roll_mean',
 's_21_roll_mean',
 's_7_roll_std',
 's_12_roll_std',
 's_14_roll_std',
 's_20_roll_std',
 's_21_roll_std']

In [7]:
# =========================
# LOAD DATA + CREATE FEATURES
# =========================

import pandas as pd

# load raw data (adjust path if needed)
df = pd.read_csv("../data/processed/your_processed_file.csv")

# OR if you used a function earlier:
# df = load_cmapss(...)

# make sure RUL already exists

# rolling function
def add_rolling_features(df, sensor_cols, window_size=5):
    df = df.sort_values(["unit", "cycle"]).copy()

    for col in sensor_cols:
        df[f"{col}_roll_mean"] = (
            df.groupby("unit")[col]
            .rolling(window=window_size, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        df[f"{col}_roll_std"] = (
            df.groupby("unit")[col]
            .rolling(window=window_size, min_periods=1)
            .std()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )

    return df


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/your_processed_file.csv'

In [ ]:
engine_ids = df_features["unit"].unique()

train_ids, val_ids = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)

train_df = df_features[df_features["unit"].isin(train_ids)].copy()
val_df = df_features[df_features["unit"].isin(val_ids)].copy()